In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import urllib.request

df = pd.read_csv('bio-pathways-network.csv')

edges = list(zip(df['Gene ID 1'], df['Gene ID 2']))

G_human = nx.Graph()
G_human.add_edges_from(edges)

print("Human nodes:", len(G_human.nodes()), "edges:", len(G_human.edges()))

urllib.request.urlretrieve(
    'http://snap.stanford.edu/deepnetbio-ismb/ipynb/yeast.edgelist',
    'yeast.edgelist'
)
yeast_file = "yeast.edgelist"
G_yeast = nx.read_edgelist(yeast_file)

print("Yeast nodes:", len(G_yeast.nodes()), "edges:", len(G_yeast.edges()))

Human nodes: 21557 edges: 342353
Yeast nodes: 6526 edges: 532180


In [2]:
from networkx.algorithms.community import modularity
from networkx.utils import py_random_state

Yeast Community Detection

Lack of data on number of communities in the yeast ppi, hence resolution is kept at default value of 1

In [3]:
yeast_grps = nx.community.louvain_communities(G_yeast, seed=42)
print(len(yeast_grps))

for community in yeast_grps:
    print(community)
    print('\n')

6
{'YDR098C-B', 'YIL127C', 'YPR144C', 'YJL080C', 'YDL167C', 'YNL014W', 'YGR271C-A', 'YIL079C', 'YER190W', 'YKL009W', 'tL(CAA)K', 'tR(UCU)K', 'YDL083C', 'YMR131C', 'YLR340W', 'YNL050C', 'YDR234W', 'YLR017W', 'YDR210C-D', 'YLR051C', 'YNL140C', 'RP23-282E10.1', 'tR(ACG)J', 'YNL069C', 'RP3-393D12.2', 'PRO1608', 'YOR293W', 'YML063W', 'YNL207W', 'YML056C', 'YNL067W', 'SPBC29A3.16', 'YLR022C', 'YNL290W', 'YPR010C-A', 'YLL035W', 'YNL082W', 'YPL001W', 'YGR049W', 'YDR447C', 'YPL037C', 'YAL038W', 'YML111W', 'YBR079C', 'YBL028C', 'YBL033C', 'YKL089W', 'YIL064W', 'YDR341C', 'YMR290C', 'YGL227W', 'YHR117W', 'YMR265C', 'YBR249C', 'YJR146W', 'YGR134W', 'YLR401C', 'YOL010W', 'YHR034C', 'YGL064C', 'HSPC131', 'YDL077C', 'RDN58-2', 'tD(GUC)L1', 'tL(GAG)G', 'DAAP-218M18.2', 'YGL232W', 'tR(ACG)D', 'YDR210C-C', 'YLR449W', 'YDL060W', 'H14', 'RP1-56L9.2', 'YBL072C', 'RP11-220B22.3', 'YDR211W', 'RP11-8N6.1', 'YJR094W-A', 'YIL069C', 'tI(UAU)D', 'YLR002C', 'YIL096C', 'YJL109C', 'YER126C', 'YJL122W', 'YDR030C', 't

Human Community Detection

Number of diseases : 519

In [ ]:
human_grps = nx.community.louvain_communities(G_human, resolution=13.25295, seed=42)
print(len(human_grps))

519


Checking accuracy of communities matched 

In [ ]:
human_df = pd.read_csv('bio-pathways-associations.csv')

human_df['Matched Community'] = None
human_df['Percentage Match'] = 0.0

for community in human_grps:
    community_set = set(community)
    for idx, row in human_df.iterrows():
        gene_set = set(map(int ,row['Associated Gene IDs'].split(', ')))
        intersection = community_set.intersection(gene_set)
        if len(intersection) > 0:
            percentage_match = len(intersection) / len(gene_set) * 100
            if percentage_match > row['Percentage Match']:
                human_df.at[idx, 'Matched Community'] = community
                human_df.at[idx, 'Percentage Match'] = percentage_match

human_df.head()
human_df.to_csv('bio-pathways-associations-with-communities.csv', index=False)

,Disease ID,Disease Name,Associated Gene IDs,Matched Community,Percentage Match
0,C0036095,Salivary Gland Neoplasms,"1462, 1612, 182, 2011, 2019, 2175, 2195, 23209...","{55808, 8704, 728577, 79369, 83468, 8720, 8722...",8.888889
1,C0033941,"Psychoses, Substance-Induced","135, 1636, 207, 2099, 2912, 2950, 3350, 3362, ...","{152579, 2054, 2055, 51208, 11270, 7180, 23566...",11.764706
2,C0043459,Zellweger Syndrome,"3295, 5189, 5190, 5192, 5193, 5194, 5195, 5567...","{5824, 11264, 5825, 373509, 5192, 5193, 201164...",33.333333
3,C0033860,Psoriasis,"100271719, 10318, 10498, 10547, 10758, 10866, ...","{307200, 3589, 3590, 2057, 3596, 3597, 3598, 5...",6.250000
4,C0027726,Nephrotic Syndrome,"1277, 1282, 1284, 2, 213, 2152, 2247, 2262, 29...","{4099, 714, 3082, 3083, 12, 5648, 5653, 6678, ...",23.809524


The below statistics show the extent each community was matched correctly by Louvain Algorithm

The maximum percentage match is only 85% and the median and average percentage match are very low, hence Louvain algorithm is not accurate at precise identification of a large number of communities

In [24]:
#max percentage match
max_match = human_df['Percentage Match'].max()
print(f'Maximum Percentage Match: {max_match}%')

#min percentage match
min_match = human_df['Percentage Match'].min()
print(f'Minimum Percentage Match: {min_match}%')

#median percentage match
median_match = human_df['Percentage Match'].median()
print(f'Median Percentage Match: {median_match}%')

#average percentage match
avg_match = human_df['Percentage Match'].mean()
print(f'Average Percentage Match: {avg_match}%')

Maximum Percentage Match: 85.0%
Minimum Percentage Match: 2.2222222222222223%
Median Percentage Match: 10.0%
Average Percentage Match: 13.529805926725906%


To do:
1. cluster distribution for yeast only
2. PCA visualization for yeast and human
3. Clustering metrics for yeast and human